# AI Core 和 Vector Core 限核功能

## 1. 功能简介

在多流场景中，如果一条 Stream 独占全部计算核，其他 Stream 即使已就绪也难以并行执行。限核功能通过约束单条 Stream 上算子的最大用核数，为其他 Stream 预留资源，从而为并行创造条件。

`npugraph_ex` 提供 **Stream 级核数配置**，可分别设置 AI Core 和 Vector Core 的使用上限。该配置表示“最大值”，并不保证算子一定使用指定数量的核。

配置值不能超过目标 AI 处理器支持的最大 AI Core 数和最大 Vector Core 数。最终收益取决于算子类型、问题规模、流间依赖和整体资源占用，应使用 Profiler 验证。

> 更多关于 AI Core 和 Vector Core 介绍参考 [AI Core/Cube Core/Vector Core 简介](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/appendix/appendix/aicore.md)，如需了解 Eager 和图模式下的控核差异请参考 [Eager 和图模式下控核介绍](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/appendix/appendix/core_limit.md)。

## 2. 使用约束

- 本功能支持的产品型号参见[使用说明](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/overview.md#%E4%BD%BF%E7%94%A8%E8%AF%B4%E6%98%8E)。
- 仅支持对 Ascend C 算子控核；对于非 Ascend C 算子暂不支持控核，并且 micro batch 多流并行场景下存在卡死可能或其他影响，不推荐使用本功能。
  - 通信类算子仅支持对 AI Vector 算子控核。
  - 对于支持[静态 Kernel 编译功能](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/npugraph%5Fex/basic/static%5Fkernel%5Fcompile.md)的算子，在 CANN 版本小于等于 CANN 8.5.0 时，在静态 kernel 和控核功能同时开启的情况下，优先保证控核功能生效，静态 kernel 功能将失效。
  - 主要适用于 micro batch 多流并行，如果存在不支持控核的算子，可能会影响多流并行效果。
  - 不支持多线程并发设置同一条流上的控核数，无法保证算子执行时的控核生效值。
- 配置核数不能超过 AI 处理器本身允许的最大核数。假设最大 AI Core 数为 `max_aicore`、最大 Vector Core 数量为 `max_vectorcore`，系统默认采用最大核数作为实际运行核数。
  您可通过 `CANN软件安装目录/<arch>-linux/data/platform_config/<soc_version>.ini` 文件查看，如下所示，说明 AI 处理器上存在 24 个 Cube Core，存在 48 个 Vector Core。
  ```ini
  [SoCInfo]
  ai_core_cnt=24
  cube_core_cnt=24
  vector_core_cnt=48
  ```

## 3. 使用方法

1. 分析模型脚本中需要指定核数的算子。
2. 使用如下 `with` 语句块配置 Stream 级核数。在被设置核数的 Stream 上执行的算子，运行时以指定核数为上限。
   ```python
   with torch.npu.npugraph_ex.scope.limit_core_num(
       op_aicore_num: int, op_vectorcore_num: int, stream: torch.npu.Stream = None
   ):
   ```
   - `op_aicore_num`：算子运行时的最大 AI Core 数，取值范围为 `[1, max_aicore]`。
   - `op_vectorcore_num`：算子运行时的最大 Vector Core 数，取值范围为 `[1, max_vectorcore]`。当 AI 处理器上仅存在 AI Core 不存在 Vector Core 时，此时仅支持取值为 0。
   - `stream`：可选参数，`torch.npu.Stream` 类型，表示需要设置核数的 Stream。默认值为 `None` 时，对进入 `with` 语句块时的当前 Stream 控核。
3. 查看配置结果。配置结果可通过 **Ascend PyTorch Profiler**（推荐 `torch_npu.profiler.profile` 接口）采集性能数据查看，详细的使用方法和结果文件介绍请参考《CANN 性能调优工具》中的“Ascend PyTorch 调优工具”章节。算子核配置结果位于 `kernel_details.csv` 中：
   - 如果是 AI Core 或 AI Vector 算子，对应的核使用的核数位于 `Block Num` 列。
   - 如果是 Mix Core 算子，主加速器使用的核数位于 `Block Num` 列，从加速器的核数位于 `Mix Block Num` 列。

## 4. 使用示例

下面的示例展示了如何在 `npugraph_ex` 后端中使用限核功能。通过 `limit_core_num` 上下文管理器，将语句块内的算子限制在指定的核数上运行，而语句块外的算子则使用默认的最大核数。

`limit_core_num` 配置的是用核上限，并不保证算子一定按该值用核，需通过 **Ascend PyTorch Profiler** 采集性能数据验证。示例将采集结果导出为 `trace.json`；在浏览器中打开 `chrome://tracing` 并加载该文件，即可在时间线中找到两个输入及计算量相同的 `Mm` 算子，对比受限算子与不受限算子的运行时间。

In [ ]:
import os
import torch
import torch_npu
from torch_npu.profiler import profile, ProfilerActivity


class CoreLimitedModel(torch.nn.Module):
    def forward(self, a, b):
        # 指定 Stream 级核数：最大 4 个 AI Core，8 个 Vector Core
        with torch.npu.npugraph_ex.scope.limit_core_num(4, 8):
            # 受核数限制的矩阵乘法
            limited_mm = torch.mm(a, b)

        # 不受核数限制的矩阵乘法，输入完全相同，确保计算量一致
        unrestricted_mm = torch.mm(a, b)

        return limited_mm, unrestricted_mm


# limit_core_num 位于 forward 内部，编译时会被记录到对应 Stream 范围。
model = CoreLimitedModel().npu()
compiled = torch.compile(model, backend="npugraph_ex", fullgraph=True, dynamic=False)

# 构造 2 个输入张量
in1 = torch.randn(1000, 1000, dtype=torch.float16).npu()
in2 = torch.randn(1000, 1000, dtype=torch.float16).npu()

# 预热一次，避免编译开销计入采集
compiled(in1, in2)

# 配置 Profiler 并指定 JSON 保存路径
prof_save_dir = "./prof_core_limit"
os.makedirs(prof_save_dir, exist_ok=True)
trace_file = os.path.join(prof_save_dir, "trace.json")

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.NPU],
    schedule=torch_npu.profiler.schedule(wait=0, warmup=0, active=1, repeat=1, skip_first=0),
) as prof:
    result = compiled(in1, in2)
    prof.step()

prof.export_chrome_trace(trace_file)

print(f"Result:\n{result}\n")
print(f"Profiling 数据已导出至文件：{trace_file}")
print(f"请在浏览器中打开 chrome://tracing，点击 'Load' 加载该 JSON 文件。")
print(f"在时间线中找到两个 Mm 算子，对比受限算子与不受限算子的运行时间。")

## 5. 关键约束与注意事项

- 主要适用于 micro batch 多流并行场景。
- 非 Ascend C 算子暂不支持控核，通信算子只支持 AI Vector 控核。
- 不支持多线程并发设置同一条流上的控核数，无法保证算子执行时的控核生效值。
- 静态 Kernel 与限核同时使用时，不同 CANN 版本可能有优先级差异（CANN <= 8.5.0 时优先控核，静态 Kernel 失效）。
- 必须通过 Profiler 的 `kernel_details.csv` 验证控核是否生效。

## 6. 课后练习

### 一、单选题

（1）【单选题】`limit_core_num` 配置的核心语义是什么？
- A. 为整个进程固定分配核数
- B. 设置指定 Stream 上算子使用 AI Core 和 Vector Core 的最大值
- C. 强制每个算子使用相同核数
- D. 关闭 NPU 上的所有并行

（2）【单选题】AI Core 算子的实际用核数在 Profiler 的 `kernel_details.csv` 中主要查看哪一列？
- A. Block Num
- B. Cache Path
- C. Input Shape
- D. Graph Break

（3）【单选题】`stream=None` 作为 `limit_core_num` 的 stream 参数时，表示什么？
- A. 不进行控核
- B. 对进入 `with` 语句块时的当前 Stream 控核
- C. 对所有进程中的全部 Stream 控核
- D. 只对默认 CPU Stream 控核

（4）【单选题】对通信类算子，当前限核功能支持哪种类型的控核？
- A. AI Cube 控核
- B. AI Vector 控核
- C. CPU Core 控核
- D. 不支持任何控核

（5）【单选题】在 CANN <= 8.5.0 时，同时开启静态 Kernel 与控核功能，通常优先保证什么生效？
- A. 静态 Kernel，控核失效
- B. 控核，静态 Kernel 失效
- C. 两者都会失效
- D. 自动开启 SuperKernel

（6）【单选题】不推荐多线程并发设置同一条 Stream 控核数的主要原因是？
- A. 无法保证算子执行时的控核生效值
- B. 会删除缓存目录
- C. 会关闭 Python 线程
- D. 会自动切换 CPU

### 二、多选题

（7）【多选题】关于限核功能的描述，正确的有哪些？
- A. 配置值是最大值，不保证算子一定用满指定核数
- B. 配置值不能超过目标 AI 处理器允许的最大核数
- C. 可通过 SoC 配置文件查看 AI Core 与 Vector Core 数量
- D. 任何非 Ascend C 算子都支持控核

（8）【多选题】限核功能更适合哪些使用方式或场景？
- A. micro batch 多流并行场景
- B. 为不同 Stream 预留资源，避免单流独占全部计算核
- C. 与 Profiler 结合验证实际的核使用情况
- D. 不分析算子和依赖关系，直接对所有模型启用

（9）【多选题】使用 `limit_core_num` 时，正确的做法有哪些？
- A. 分别设置 `op_aicore_num` 与 `op_vectorcore_num` 的合法值
- B. 在目标 Stream 的上下文中配置需要受限的算子
- C. 对 Mix Core 算子同时关注 `Block Num` 和 `Mix Block Num`
- D. 仅根据配置值推断控核已经生效

（10）【多选题】下列哪些限制或注意事项是正确的？
- A. 非 Ascend C 算子暂不支持控核
- B. 通信类算子仅支持 AI Vector 控核
- C. 静态 Kernel 和控核的优先级会受 CANN 版本影响
- D. 多线程可无风险地并发设置同一 Stream 的控核数

**运行以下代码单元查看参考答案与解析。**


In [ ]:
import os
answer_path = "answer/04.04_answer.txt"
if os.path.exists(answer_path):
    with open(answer_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("答案文件未找到，请检查 answer 目录。")
